# Training makna + tingkat (training-makna-tingkat-v1)

Hasil diskusi `2026-11-22-perbaikan-sisi-training-data.md`. Pertanyaan: apakah model dasar yang lebih besar
(NLLB-200-distilled-1.3B) menjaga makna lebih baik dengan paket data demo, dan apakah konverter Krama→Krama Alus
memperbaiki arah hormat.

- **MODE = "valid"**: hanya train + valid. Demo v2 (pembanding) → pagar zero-shot 600M vs 1.3B → preflight memori 1.3B →
  `nllb600_base` → `nllb13_base` (bila lolos) → konverter input-bersih. Ekspor prediksi valid + manifest.
- **MODE = "test"**: dibuat otomatis oleh `scripts/pilih_training_makna.py` setelah pilihan dikunci.

In [ ]:
MODE = "valid"
PILIHAN = None  # diisi otomatis untuk MODE test

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sacrebleu", "sentencepiece"], check=False)

import glob, json, time, random, os, hashlib, platform, gc, math
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import numpy as np, pandas as pd, torch, sacrebleu, transformers
import torch.nn.functional as F
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)
from transformers.optimization import Adafactor

SEED, MAXLEN, LAM, NBEST = 42, 64, 2.0, 8
EPOCHS, OS, BLOCK, LR, WARMUP, CTRL_W = 15, 3, 16, 5e-5, 0.1, 30.0
GPU_CAP_H = 6.0
ARMS = {"nllb600_base": "facebook/nllb-200-distilled-600M", "nllb13_base": "facebook/nllb-200-distilled-1.3B"}
SL, TL = "ind_Latn", "jav_Latn"
OUT = "/kaggle/working"
dev = "cuda"
LEVELS = ["ngoko", "krama", "krama_alus"]
LID = {l: i for i, l in enumerate(LEVELS)}
TAG = {l: f"<{l}>" for l in LEVELS}
INIT_WORDS = {"ngoko": "kowe aku mangan lunga turu", "krama": "sampeyan kula nedha kesah tilem",
              "krama_alus": "panjenenganipun dalem dhahar tindak sare"}
T0 = time.time()
log = lambda *a: print(f"[{time.time()-T0:6.0f}s]", *a, flush=True)
M = {"mode": MODE, "seed": SEED, "hparams": {"epochs": EPOCHS, "os": OS, "block": BLOCK, "lr": LR, "warmup": WARMUP,
     "ctrl_w": CTRL_W, "lambda": LAM, "nbest": NBEST, "maxlen": MAXLEN, "gpu_cap_h": GPU_CAP_H,
     "optimizer": "Adafactor(relative_step=False, scale_parameter=False, warmup_init=False, clip_threshold=1.0, weight_decay=0.01)"},
     "versions": {"python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__,
                  "sacrebleu": sacrebleu.__version__},
     "gpu": torch.cuda.get_device_name(0), "inputs": {}, "gates": {}, "arms": {}}


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


sha256 = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
chrf = lambda h, r: sacrebleu.corpus_chrf(list(h), [list(r)], word_order=2).score


def save_manifest():
    M["elapsed_s"] = round(time.time() - T0)
    json.dump(M, open(f"{OUT}/manifest_{MODE}.json", "w"), indent=1, ensure_ascii=False)


def free():
    gc.collect(); torch.cuda.empty_cache()

## Preflight: data, model demo v2, reranker

In [ ]:
cfgs = glob.glob("/kaggle/input/**/model/demo_config.json", recursive=True)
assert len(cfgs) == 1, cfgs
DEMO_DIR = os.path.dirname(cfgs[0]); V2_DIR = os.path.dirname(DEMO_DIR)
DCFG = json.load(open(cfgs[0]))
assert DCFG["variant"] == "m2m_base" and DCFG["nbest"] == NBEST and float(DCFG["rerank_lambda"]) == LAM
M["inputs"]["demo_config_sha256"] = sha256(cfgs[0])


def load_split(n):
    ps = [p for p in glob.glob(f"/kaggle/input/**/{n}.csv", recursive=True) if not p.startswith(V2_DIR)]
    assert len(ps) == 1, (n, ps)
    M["inputs"][f"{n}_csv_sha256"] = sha256(ps[0])
    d = pd.read_csv(ps[0]); d = d[d["level"].isin(LEVELS)].copy(); d["label"] = d["level"].map(LID)
    return d.reset_index(drop=True)


tr_raw = load_split("train")
tr, va = tr_raw, load_split("valid")
te = load_split("test") if MODE == "test" else None      # MODE valid tidak membaca test
assert (len(tr), len(va)) == (1395, 175), (len(tr), len(va))
assert not set(tr.group) & set(va.group) and not set(tr.indonesian) & set(va.indonesian)
assert va[va.level == "krama_alus"].group.nunique() == 27
tag_src = lambda s, l: f"{TAG[l]} {s}"

rr_dir = os.path.join(DEMO_DIR, DCFG["reranker_dir"])
rr_tok = AutoTokenizer.from_pretrained(rr_dir)
rr_mod = AutoModelForSequenceClassification.from_pretrained(rr_dir).float().to(dev).eval()


def cls_logprobs(ctok, cmod, sents):
    out = []
    for i in range(0, len(sents), 64):
        enc = ctok(sents[i:i + 64], return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN).to(dev)
        with torch.no_grad():
            out.append(F.log_softmax(cmod(**enc).logits.float(), -1).cpu())
    return torch.cat(out).numpy().astype(np.float64)


rr_lp = lambda s: cls_logprobs(rr_tok, rr_mod, s)
log("data", len(tr), len(va), "| demo", DEMO_DIR, "|", M["gpu"])

## Juri valid (DistilBERT, resep v2, seed sebelum inisialisasi) — berbeda dari reranker

In [ ]:
seed_all(SEED)
j_tok = AutoTokenizer.from_pretrained("w11wo/javanese-distilbert-small")
j_mod = AutoModelForSequenceClassification.from_pretrained("w11wo/javanese-distilbert-small", num_labels=3).to(dev)
_opt = torch.optim.AdamW(j_mod.parameters(), lr=5e-5)
X, Y = tr["javanese"].tolist(), tr["label"].tolist()
for _ in range(4):
    j_mod.train(); order = list(range(len(X))); random.shuffle(order)
    for i in range(0, len(order), 32):
        b = order[i:i + 32]
        enc = j_tok([X[k] for k in b], return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN).to(dev)
        loss = j_mod(**enc, labels=torch.tensor([Y[k] for k in b], device=dev)).loss
        _opt.zero_grad(); loss.backward(); _opt.step()
j_mod.eval(); del _opt
judge = lambda s: [LEVELS[i] for i in cls_logprobs(j_tok, j_mod, list(s)).argmax(-1)]
M["valid_judge_acc_on_refs"] = round(float(np.mean(np.array(judge(va.javanese)) == va.level.values)), 4)
log("juri valid akurasi pada referensi", M["valid_judge_acc_on_refs"])


def valid_score(hyps):
    lab = np.array(judge(hyps)); y = va.level.values
    c = chrf(hyps, va.javanese); a = float((lab == y).mean())
    return c + CTRL_W * a, round(c, 2), round(a, 4)

## Fungsi MT (resep v2)

In [ ]:
def seq_scores(tok, model, srcs, cands, src_lang, chunk=16):
    tok.src_lang = src_lang; out = []
    for i in range(0, len(srcs), chunk):
        enc = tok(srcs[i:i + chunk], text_target=cands[i:i + chunk], return_tensors="pt",
                  padding=True, truncation=True, max_length=MAXLEN).to(dev)
        labels = enc.pop("labels"); labels[labels == tok.pad_token_id] = -100
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            logits = model(**enc, labels=labels).logits
        nll = F.cross_entropy(logits.float().transpose(1, 2), labels, ignore_index=-100, reduction="none")
        m = (labels != -100).float()
        out += (-(nll * m).sum(1) / m.sum(1).clamp(min=1)).cpu().tolist()
        del logits, nll
    return np.array(out, dtype=np.float64)


def generate(tok, model, bos, srcs, src_lang, beams, k=1, bs=16):
    model.eval(); tok.src_lang = src_lang; out = [None] * len(srcs)
    order = sorted(range(len(srcs)), key=lambda i: len(srcs[i]))
    for i in range(0, len(order), bs):
        idx = order[i:i + bs]
        enc = tok([srcs[j] for j in idx], return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN).to(dev)
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            g = model.generate(**enc, forced_bos_token_id=bos, num_beams=beams, num_return_sequences=k,
                               max_new_tokens=MAXLEN)
        txt = tok.batch_decode(g, skip_special_tokens=True)
        for n, j in enumerate(idx):
            out[j] = txt[n * k:(n + 1) * k] if k > 1 else txt[n]
    free()
    return out


def rerank_decode(tok, model, bos, srcs, levels, src_lang):
    """beam8, 8 kandidat, skor = log-prob rata-rata teacher-forced + LAM * log P_reranker(tingkat diminta)."""
    nb = generate(tok, model, bos, srcs, src_lang, NBEST, k=NBEST, bs=8)
    flat = [c for cs in nb for c in cs]
    sc = seq_scores(tok, model, [s for s, cs in zip(srcs, nb) for _ in cs], flat, src_lang)
    lp = rr_lp(flat); res = []; p = 0
    for cs, l in zip(nb, levels):
        k = len(cs); tot = sc[p:p + k] + LAM * lp[p:p + k, LID[l]]
        res.append(cs[int(np.argmax(tot))]); p += k
    return res


def export_valid(name, hyps):
    d = va[["index", "group", "level", "indonesian", "javanese"]].copy()
    d["hyp"] = hyps; d["juri_valid"] = judge(hyps)
    d.to_csv(f"{OUT}/valid_{name}.csv", index=False)
    m = {lv: round(chrf(d.hyp[d.level == lv], d.javanese[d.level == lv]), 2) for lv in LEVELS}
    m["all"] = round(chrf(d.hyp, d.javanese), 2)
    m["KA_juri_benar"] = int(((d.level == "krama_alus") & (d.juri_valid == "krama_alus")).sum())
    m["sha256"] = sha256(f"{OUT}/valid_{name}.csv")
    log("valid", name, m)
    return m

## Demo v2 (pembanding, dijalankan di kernel yang sama)

In [ ]:
if MODE == "valid":
    d_tok = AutoTokenizer.from_pretrained(DEMO_DIR, src_lang=DCFG["src_lang"], tgt_lang=DCFG["tgt_lang"])
    d_mod = AutoModelForSeq2SeqLM.from_pretrained(DEMO_DIR).float().to(dev).eval()
    d_bos = d_tok.get_lang_id(DCFG["tgt_lang"])
    hyp = rerank_decode(d_tok, d_mod, d_bos, [tag_src(s, l) for s, l in zip(va.indonesian, va.level)],
                        va.level.tolist(), DCFG["src_lang"])
    M["arms"]["demo"] = export_valid("demo", hyp)
    M["gates"]["reproduksi_demo"] = {"chrF_all": M["arms"]["demo"]["all"], "target_all": 62.66,
                                     "chrF_KA": M["arms"]["demo"]["krama_alus"], "target_KA": 68.83,
                                     "ok": abs(M["arms"]["demo"]["all"] - 62.66) < 0.05 and abs(M["arms"]["demo"]["krama_alus"] - 68.83) < 0.05}
    log("reproduksi demo", M["gates"]["reproduksi_demo"])
    del d_mod; free(); save_manifest()

## Pagar zero-shot 600M vs 1.3B (74 Ngoko valid, tanpa tag, beam4)

In [ ]:
def boot_p5(h_a, h_b, refs, groups, n=2000, seed=SEED):
    """persentil ke-5 dari selisih corpus chrF++ (b − a), resampel per grup berpasangan."""
    groups = np.asarray(groups); ug = np.unique(groups); idx = {g: np.where(groups == g)[0] for g in ug}
    rng = np.random.default_rng(seed); h_a, h_b, refs = map(np.asarray, (h_a, h_b, refs)); diffs = []
    for _ in range(n):
        s = np.concatenate([idx[g] for g in rng.choice(ug, len(ug))])
        diffs.append(chrf(h_b[s], refs[s]) - chrf(h_a[s], refs[s]))
    return float(np.percentile(diffs, 5)), float(np.mean(diffs))


def load_base(repo):
    tok = AutoTokenizer.from_pretrained(repo, src_lang=SL, tgt_lang=TL)
    model = AutoModelForSeq2SeqLM.from_pretrained(repo)
    return tok, model, tok.convert_tokens_to_ids(TL)


if MODE == "valid":
    ng = va[va.level == "ngoko"].reset_index(drop=True)
    assert len(ng) == 74
    zs = {}
    for arm, repo in ARMS.items():
        tok, model, bos = load_base(repo); model.to(dev)
        zs[arm] = generate(tok, model, bos, ng.indonesian.tolist(), SL, beams=4)
        M["gates"][f"zeroshot_ngoko_{arm}"] = round(chrf(zs[arm], ng.javanese), 2)
        del model; free()
    p5, mean = boot_p5(zs["nllb600_base"], zs["nllb13_base"], ng.javanese, ng.group)
    M["gates"]["zeroshot_p5_13_minus_600"] = round(p5, 3); M["gates"]["zeroshot_mean_13_minus_600"] = round(mean, 3)
    M["gates"]["zeroshot_lolos"] = bool(p5 > 0)
    pd.DataFrame({"group": ng.group, "indonesian": ng.indonesian, "reference": ng.javanese, **zs}).to_csv(f"{OUT}/zeroshot_ngoko_valid.csv", index=False)
    log("pagar zero-shot", {k: v for k, v in M["gates"].items() if k.startswith("zeroshot")})
    save_manifest()

## Training satu arm (paket demo; Adafactor; fp16 AMP; gradient checkpointing; akumulasi per blok v2)

In [ ]:
def load_tagged(repo):
    tok, model, bos = load_base(repo)
    tok.add_tokens(list(TAG.values()), special_tokens=True)
    model.resize_token_embeddings(len(tok))
    emb = model.get_input_embeddings().weight
    with torch.no_grad():
        for l, words in INIT_WORDS.items():
            ids = tok(words, add_special_tokens=False)["input_ids"]
            emb[tok.convert_tokens_to_ids(TAG[l])] = emb[ids].mean(0)
    return tok, model.to(dev), bos


def honor_pairs():
    pairs = []
    for _, g in tr_raw.groupby("group"):
        jv = dict(zip(g["level"], g["javanese"]))
        if "krama" in jv and "krama_alus" in jv and jv["krama"] != jv["krama_alus"]:
            pairs.append((f"{TAG['krama_alus']} {jv['krama']}", jv["krama_alus"]))
    return pairs


def make_opt(model, n_updates):
    opt = Adafactor(model.parameters(), lr=LR, relative_step=False, scale_parameter=False, warmup_init=False,
                    clip_threshold=1.0, weight_decay=0.01)
    return opt, get_linear_schedule_with_warmup(opt, int(WARMUP * n_updates), n_updates)


def block_step(tok, model, opt, sched, scaler, src_lang, block, mb):
    """Satu update untuk satu blok v2: loss = jumlah NLL token / jumlah token non-padding di blok."""
    tok.src_lang = src_lang
    encs = [tok([s for s, _ in block[i:i + mb]], text_target=[t for _, t in block[i:i + mb]], return_tensors="pt",
                padding=True, truncation=True, max_length=MAXLEN) for i in range(0, len(block), mb)]
    n_tok = sum(int((e["labels"] != tok.pad_token_id).sum()) for e in encs)
    opt.zero_grad(set_to_none=True); tot = 0.0
    for e in encs:
        e = e.to(dev); labels = e.pop("labels"); labels[labels == tok.pad_token_id] = -100
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(**e, labels=labels).logits
        loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), labels.view(-1), ignore_index=-100,
                               reduction="sum") / n_tok
        scaler.scale(loss).backward(); tot += loss.item(); del logits, loss
    scaler.step(opt); scaler.update(); sched.step()
    return tot


def make_blocks(id_pairs, jv_pairs):
    a, b = id_pairs[:], jv_pairs[:]
    random.shuffle(a); random.shuffle(b)
    bt = [(SL, a[i:i + BLOCK]) for i in range(0, len(a), BLOCK)] + [(TL, b[i:i + BLOCK]) for i in range(0, len(b), BLOCK)]
    random.shuffle(bt); return bt


base_pairs = [(tag_src(s, l), t, l) for s, l, t in zip(tr["indonesian"], tr["level"], tr["javanese"])]
ID_PAIRS = [(s, t) for s, t, _ in base_pairs] + [(s, t) for s, t, l in base_pairs if l == "krama_alus"] * (OS - 1)
JV_PAIRS = honor_pairs()
N_BLOCKS = math.ceil(len(ID_PAIRS) / BLOCK) + math.ceil(len(JV_PAIRS) / BLOCK)
M["hparams"].update({"id_pairs": len(ID_PAIRS), "jv_pairs": len(JV_PAIRS), "blocks_per_epoch": N_BLOCKS,
                     "updates": N_BLOCKS * EPOCHS})
log("pasangan ID->JV", len(ID_PAIRS), "JV->JV", len(JV_PAIRS), "blok/epoch", N_BLOCKS)


def preflight_mem(repo, mb):
    """Dua update pada blok terpanjang (64 token), lalu satu evaluasi greedy dan beam8. Kembalikan (ok, info)."""
    seed_all(SEED); tok, model, bos = load_tagged(repo)
    model.gradient_checkpointing_enable(); model.config.use_cache = False; model.train()
    opt, sched = make_opt(model, 10); scaler = torch.amp.GradScaler("cuda")
    longest = sorted(ID_PAIRS, key=lambda p: -len(p[0]) - len(p[1]))[:BLOCK]
    torch.cuda.reset_peak_memory_stats(); info = {"mb": mb}
    try:
        ts = []
        for _ in range(3):
            t = time.time(); loss = block_step(tok, model, opt, sched, scaler, SL, longest, mb); torch.cuda.synchronize()
            ts.append(time.time() - t)
            assert math.isfinite(loss), f"loss tidak finite: {loss}"
        assert all(torch.isfinite(p).all() for p in model.parameters() if p.requires_grad)
        model.config.use_cache = True
        generate(tok, model, bos, [p[0] for p in longest[:8]], SL, beams=NBEST, k=NBEST, bs=8)
        info.update({"ok": True, "detik_per_update_terpanjang": round(float(np.mean(ts[1:])), 3),
                     "peak_alloc_gib": round(torch.cuda.max_memory_allocated() / 2**30, 2),
                     "peak_reserved_gib": round(torch.cuda.max_memory_reserved() / 2**30, 2)})
    except torch.cuda.OutOfMemoryError as e:
        info.update({"ok": False, "alasan": "OOM", "pesan": str(e)[:200]})
    del model, opt, sched, scaler; free()
    return info["ok"], info


def train_arm(name, repo, mb):
    seed_all(SEED); tok, model, bos = load_tagged(repo)
    model.gradient_checkpointing_enable()
    opt, sched = make_opt(model, N_BLOCKS * EPOCHS); scaler = torch.amp.GradScaler("cuda")
    best, best_state, hist = -1e9, None, []
    va_src = [tag_src(s, l) for s, l in zip(va.indonesian, va.level)]
    for ep in range(EPOCHS):
        model.train(); model.config.use_cache = False; tot = 0.0; t = time.time()
        for src_lang, b in make_blocks(ID_PAIRS, JV_PAIRS):
            tot += block_step(tok, model, opt, sched, scaler, src_lang, b, mb)
        model.config.use_cache = True
        sc, c, a = valid_score(generate(tok, model, bos, va_src, SL, beams=1))
        hist.append({"epoch": ep + 1, "loss": round(tot / N_BLOCKS, 4), "valid_chrF++": c, "valid_ctrl": a,
                     "score": round(sc, 2), "detik": round(time.time() - t)})
        log(name, hist[-1])
        if sc > best:
            best = sc; best_state = {k: x.detach().to("cpu", copy=True) for k, x in model.state_dict().items()}
    model.load_state_dict(best_state); del best_state, opt, sched, scaler; free()
    hyp = rerank_decode(tok, model, bos, va_src, va.level.tolist(), SL)
    res = export_valid(name, hyp)
    res.update({"history": hist, "best_epoch": max(hist, key=lambda h: h["score"])["epoch"], "mb": mb})
    mdir = f"{OUT}/model_{name}"
    model.half().save_pretrained(mdir); tok.save_pretrained(mdir)
    json.dump({"arm": name, "base": repo, "src_lang": SL, "tgt_lang": TL, "tags": TAG, "nbest": NBEST,
               "rerank_lambda": LAM}, open(f"{mdir}/arm_config.json", "w"), indent=1)
    res["model_sha256"] = {f: sha256(os.path.join(mdir, f)) for f in sorted(os.listdir(mdir)) if f.endswith(".safetensors")}
    del model; free()
    return res

## Jalankan arm (MODE valid)

In [ ]:
if MODE == "valid":
    mb, run13 = 2, M["gates"]["zeroshot_lolos"]
    if run13:
        ok, info = preflight_mem(ARMS["nllb13_base"], 2)
        if not ok:
            ok, info = preflight_mem(ARMS["nllb13_base"], 1)
        M["gates"]["preflight_13"] = info
        if ok:
            mb = info["mb"]
            # proyeksi kasar: update terpanjang × jumlah update × 0,7 (blok rata-rata lebih pendek) + evaluasi
            proj_h = info["detik_per_update_terpanjang"] * N_BLOCKS * EPOCHS * 0.7 / 3600 + 0.5
            M["gates"]["proyeksi_jam_13"] = round(proj_h, 2)
            run13 = (time.time() - T0) / 3600 + proj_h + 1.0 <= GPU_CAP_H
            if not run13:
                M["gates"]["alasan_lewati_13"] = f"proyeksi {proj_h:.2f} jam melewati pagu {GPU_CAP_H} jam"
        else:
            run13 = False; M["gates"]["alasan_lewati_13"] = "preflight memori gagal (mb 2 dan 1)"
    else:
        M["gates"]["alasan_lewati_13"] = "pagar zero-shot tidak lolos"
    M["gates"]["jalankan_13"] = bool(run13); M["hparams"]["mb"] = mb
    save_manifest(); log("gerbang 1.3B", run13, "mb", mb)

    M["arms"]["nllb600_base"] = train_arm("nllb600_base", ARMS["nllb600_base"], mb); save_manifest()
    if run13:
        M["arms"]["nllb13_base"] = train_arm("nllb13_base", ARMS["nllb13_base"], mb); save_manifest()

## Konverter input-bersih Krama → Krama Alus (diagnosis; jadwal tetap, checkpoint epoch terakhir)

In [ ]:
if MODE == "valid":
    seed_all(SEED)
    tok, model, bos = load_tagged(ARMS["nllb600_base"]); model.gradient_checkpointing_enable()
    nb = math.ceil(len(JV_PAIRS) / BLOCK)
    opt, sched = make_opt(model, nb * EPOCHS); scaler = torch.amp.GradScaler("cuda")
    for ep in range(EPOCHS):
        model.train(); model.config.use_cache = False; b = JV_PAIRS[:]; random.shuffle(b); tot = 0.0
        for i in range(0, len(b), BLOCK):
            tot += block_step(tok, model, opt, sched, scaler, TL, b[i:i + BLOCK], mb)
        log("konverter epoch", ep + 1, "loss", round(tot / nb, 4))
    model.config.use_cache = True
    p = va.pivot_table(index="group", columns="level", values="javanese", aggfunc="first").dropna(subset=["krama", "krama_alus"])
    src_id = va.drop_duplicates("group").set_index("group").indonesian
    assert len(p) == 26
    outk = generate(tok, model, bos, [f"{TAG['krama_alus']} {k}" for k in p.krama], TL, beams=1)
    kv = pd.DataFrame({"group": p.index, "indonesian": src_id.loc[p.index].values, "krama_ref": p.krama.values,
                       "krama_alus_ref": p.krama_alus.values, "salinan_krama": p.krama.values, "konverter": outk})
    kv.to_csv(f"{OUT}/konverter_valid.csv", index=False)
    M["konverter"] = {"n": len(kv), "chrF_salinan": round(chrf(kv.salinan_krama, kv.krama_alus_ref), 2),
                      "chrF_konverter": round(chrf(kv.konverter, kv.krama_alus_ref), 2),
                      "juri_KA_salinan": int(sum(x == "krama_alus" for x in judge(kv.salinan_krama))),
                      "juri_KA_konverter": int(sum(x == "krama_alus" for x in judge(kv.konverter)))}
    log("konverter", M["konverter"])
    del model, opt, sched, scaler; free(); save_manifest()
    log("SELESAI run valid. Lanjutkan: python scripts/pilih_training_makna.py seleksi")

## MODE test (hanya setelah pilihan dikunci)

In [ ]:
if MODE == "test":
    assert PILIHAN is not None and PILIHAN["arm"] != "demo"
    M["pilihan"] = PILIHAN
    mcfgs = glob.glob(f"/kaggle/input/**/model_{PILIHAN['arm']}/arm_config.json", recursive=True)
    assert len(mcfgs) == 1, mcfgs
    mdir = os.path.dirname(mcfgs[0])
    for f, h in PILIHAN["model_sha256"].items():
        assert sha256(os.path.join(mdir, f)) == h, f"checkpoint {f} berubah"
    u = sorted(te.indonesian.unique()); req = [(s, l) for s in u for l in LEVELS]; assert len(req) == 225
    systems = {"demo": (DEMO_DIR, DCFG["src_lang"], DCFG["tgt_lang"], "m2m"), "pilihan": (mdir, SL, TL, "nllb")}
    M["test_chrF++"] = {}
    for name, (d, sl, tl, kind) in systems.items():
        tok = AutoTokenizer.from_pretrained(d, src_lang=sl, tgt_lang=tl)
        model = AutoModelForSeq2SeqLM.from_pretrained(d).float().to(dev).eval()
        bos = tok.get_lang_id(tl) if kind == "m2m" else tok.convert_tokens_to_ids(tl)
        ht = rerank_decode(tok, model, bos, [tag_src(s, l) for s, l in zip(te.indonesian, te.level)], te.level.tolist(), sl)
        hc = rerank_decode(tok, model, bos, [tag_src(s, l) for s, l in req], [l for _, l in req], sl)
        pred = te.copy(); pred["hyp"] = ht; pred.to_csv(f"{OUT}/pred_{name}.csv", index=False)
        pd.DataFrame({"indonesian": [s for s, _ in req], "requested": [l for _, l in req], "output": hc}).to_csv(
            f"{OUT}/control_{name}.csv", index=False)
        M["test_chrF++"][name] = {lv: round(chrf(pred.hyp[pred.level == lv], pred.javanese[pred.level == lv]), 2) for lv in LEVELS}
        del model; free()
    v2 = pd.read_csv(os.path.join(V2_DIR, "pred_m2m_base_rerank2.0.csv"))
    M["reproduksi_demo_test_identik"] = f"{int((v2.hyp.values == pd.read_csv(f'{OUT}/pred_demo.csv').hyp.values).sum())}/{len(v2)}"
    save_manifest(); log("SELESAI run test", M["test_chrF++"], M["reproduksi_demo_test_identik"])